## 1. Setup

In [60]:
import os
import sys
import random
import time
import pandas as pd
import ollama
from pathlib import Path
from datasets import load_dataset


In [61]:
# LOCAL or COLAB, detect we must.
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print('Running in COLAB, we are.')
    from google.colab import drive
    drive.mount('/content/drive')
    input_dir = Path('')
    output_dir = Path('')
else:
    print('Running locally, we are.')
    # Set the input and output folders, you must.
    input_dir = Path('../data/')
    output_dir = Path('/Users/aliceberta/Development/NLP/data')
    # Create the folders if missing, we will.
    output_dir.mkdir(parents=True, exist_ok=True)

INPUT_DIR, OUTPUT_DIR = input_dir, output_dir
print(f'Input: {INPUT_DIR} | Output: {OUTPUT_DIR}')


Running locally, we are.
Input: ../data | Output: /Users/aliceberta/Development/NLP/data


In [62]:
# The path to the package directory, define we must.
package_parent_dir = os.path.abspath('./')
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Import the client package, we must.
from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

# The credentials for the server, load we do.
username = os.getenv('USER')
pwd      = os.getenv('PASSWORD')
mail     = os.getenv('MAIL')
API_URL  = 'http://131.175.15.22:51111/'

# A client instance, create and authenticate we must.
client = MillionaireClient(API_URL)
try:
    user = client.login(username, pwd)
    print(f'Login successful. Welcome, {user.username}!')
except AuthenticationError as e:
    print(f'Login failed: {e}')


Login successful. Welcome, Alisss!


## 2. Explore Available Competitions

In [63]:
# Known, the competitions are. List them we must.
competitions = client.competitions.list_all()
for comp in competitions:
    print(f'ID: {comp.id} | Name: {comp.name} | Levels: {comp.max_levels} | Questions: {comp.question_count}')
    if comp.description:
        print(f'   {comp.description}')


ID: 0 | Name: Entertainment | Levels: 15 | Questions: 843
   Music, Movies, Celebrities and more
ID: 1 | Name: Ancient History and Politics | Levels: 15 | Questions: 454
   The Roman Empire, The Greeks, and more
ID: 2 | Name: Science and Nature | Levels: 15 | Questions: 5395
   Chemistry, Biology, Physics and similar subjects
ID: 3 | Name: Maths | Levels: 15 | Questions: 390
   Mathematics and Statistics from High School and College


## 3. Game Loop Utilities

The `play_and_log()` function handles a full game session: it receives a strategy function, loops through questions, submits answers, and logs every result. The `evaluate_strategy()` function runs multiple games to get reliable statistics. The `summarize()` function produces a summary row for comparison tables.

In [64]:
# Chosen, the competition to play in must be.
COMPETITION_ID = 1
TOTAL_QUESTIONS = 15


def play_and_log(client, competition_id, strategy_fn, strategy_name='random'):
    """Play a full game session and log each answer."""
    results = []
    print(f'\n=== Strategy: {strategy_name} | Competition ID: {competition_id} ===')
    game = client.game.start(competition_id)
    print(f'Session ID: {game.session_id}')

    while game.in_progress:
        q = game.current_question
        if not q:
            break

        print(f'\n[Level {game.current_level}] {q.text}')
        for opt in q.options:
            print(f'  [{opt.id}] {opt.text}')

        t_start = time.time()
        chosen_id = strategy_fn(q)
        elapsed = time.time() - t_start

        chosen_text = next(o.text for o in q.options if o.id == chosen_id)
        print(f'  --> Choice: "{chosen_id}" : "{chosen_text}" (in {elapsed:.2f}s)')

        result = game.answer(chosen_id)

        results.append({
            'game':             None,
            'level':            game.current_level,
            'question':         q.text,
            'chosen':           chosen_text,
            'correct':          result.correct,
            'timed_out':        result.timed_out,
            'earned':           result.earned_amount,
            'strategy':         strategy_name,
            'response_time_s':  round(elapsed, 3),
        })

        status = '✅ CORRECT' if result.correct else ('⏰ TIMEOUT' if result.timed_out else '❌ WRONG')
        print(f'  {status} | Earned: {result.earned_amount}')

        if result.game_over:
            break

    print(f'\n=== Game over | Total earned: {game.earned_amount} ===')
    return results


def evaluate_strategy(client, competition_id, strategy_fn, strategy_name, n_games=5):
    """Run multiple games and return a combined DataFrame."""
    comp = get_competition_info(client, competition_id)
    all_logs = []
    for i in range(n_games):
        print(f'\n--- Game {i+1}/{n_games} ---')
        log = play_and_log(client, competition_id, strategy_fn, strategy_name)
        for row in log:
            row['game'] = i + 1
        append_to_log(log, competition_id, comp.name, i + 1, strategy_name)
        all_logs.extend(log)
    return pd.DataFrame(all_logs)


def summarize(df):
    """Summarize a results DataFrame into a single-row dict."""
    per_game_accuracy = df.groupby('game')['correct'].sum() / TOTAL_QUESTIONS
    return {
        'Strategy':        df['strategy'].iloc[0],
        'Games':           df['game'].nunique(),
        'Total questions': len(df),
        'Accuracy':        f"{per_game_accuracy.mean():.0%}",
        'Avg level':       f"{df.groupby('game')['level'].max().mean():.1f}",
        'Avg earned':      f"{df.groupby('game')['earned'].last().mean():.0f}",
        'Avg time (s)':    f"{df['response_time_s'].mean():.3f}",
    }


def get_competition_info(client, competition_id):
    competitions = client.competitions.list_all()
    for comp in competitions:
        if comp.id == competition_id:
            return comp
    raise ValueError(f'Competition ID {competition_id} not found.')


print('Game loop utilities loaded. Ready they are.')


Game loop utilities loaded. Ready they are.


In [ ]:
import json
import datetime

LOG_FILE = Path('runs_log.json')
print(f'Log file path: {LOG_FILE.resolve()}')


def _recalc_summary(games):
    """Ricalcola i summary aggregati per una lista di game records."""
    return {
        'avg_accuracy':      round(sum(g['accuracy'] for g in games) / len(games), 4),
        'avg_earned':        round(sum(g['earned'] for g in games) / len(games), 2),
        'avg_reached_level': round(sum(g['reached_level'] for g in games) / len(games), 2),
    }


def append_to_log(log_rows, competition_id, competition_name, game_num, strategy_name):
    """Aggiunge un record di partita al log JSON nested per strategia e competizione."""
    df_game = pd.DataFrame(log_rows)
    correct = int(df_game['correct'].sum())
    record = {
        'game':              int(game_num),
        'timestamp':         datetime.datetime.now().isoformat(timespec='seconds'),
        'correct':           correct,
        'accuracy':          round(correct / TOTAL_QUESTIONS, 4),
        'earned':            float(df_game['earned'].iloc[-1]) if len(df_game) > 0 else 0.0,
        'reached_level':     int(df_game['level'].max()) if len(df_game) > 0 else 0,
        'response_time_avg': round(float(df_game['response_time_s'].mean()), 3),
    }

    data = json.loads(LOG_FILE.read_text()) if LOG_FILE.exists() else {}

    data.setdefault(strategy_name, {}).setdefault(competition_name, {'games': []})
    data[strategy_name][competition_name]['games'].append(record)
    data[strategy_name][competition_name].update(
        _recalc_summary(data[strategy_name][competition_name]['games'])
    )

    LOG_FILE.write_text(json.dumps(data, indent=2))
    n = len(data[strategy_name][competition_name]['games'])
    print(f'  [log] {strategy_name} | {competition_name} | game {game_num} saved ({n} games total)')


print('append_to_log() ready.')


In [ ]:
# Migra il vecchio log (lista piatta) al nuovo formato nested.
# Esegui una volta sola — se il file è già nel nuovo formato, non fa nulla.
if LOG_FILE.exists():
    raw = json.loads(LOG_FILE.read_text())
    if isinstance(raw, list):
        nested = {}
        for entry in raw:
            s = entry['strategy']
            c = entry['competition_name']
            nested.setdefault(s, {}).setdefault(c, {'games': []})
            nested[s][c]['games'].append({
                'game':              entry['game'],
                'timestamp':         entry['timestamp'],
                'correct':           entry['correct'],
                'accuracy':          entry['accuracy'],
                'earned':            entry['earned'],
                'reached_level':     entry['reached_level'],
                'response_time_avg': entry['response_time_avg'],
            })
        for s in nested:
            for c in nested[s]:
                nested[s][c].update(_recalc_summary(nested[s][c]['games']))
        LOG_FILE.write_text(json.dumps(nested, indent=2))
        print(f'Migrazione completata. Strategie: {list(nested.keys())}')
    else:
        print('Il log è già nel nuovo formato, nessuna migrazione necessaria.')
else:
    print('Nessun log esistente da migrare.')


## 4. Random Baseline

We start with the simplest possible strategy: answer every question at random. This gives us a lower-bound reference — any model that cannot beat random guessing is not useful. With 4 options per question, expected accuracy is 25%.

In [66]:
def random_strategy(question):
    # The force of randomness, use we will.
    return random.choice(question.options).id


N_GAMES = 5
df_random = evaluate_strategy(client, COMPETITION_ID, random_strategy, 'random', N_GAMES)



--- Game 1/5 ---

=== Strategy: random | Competition ID: 1 ===
Session ID: 87086

[Level 1] Which of the following best describes the Temple of Olympian Zeus in Athens?
  [0] It was primarily dedicated to the god Poseidon.
  [1] It was the largest temple in the ancient world during its completion.
  [2] It was completed in the 6th century BC.
  [3] It was never used after its completion.
  --> Choice: "3" : "It was never used after its completion." (in 0.00s)
  ❌ WRONG | Earned: 0

=== Game over | Total earned: 0 ===
  [log] game 1 saved → runs_log.json (1 records total)

--- Game 2/5 ---

=== Strategy: random | Competition ID: 1 ===
Session ID: 87087

[Level 1] Which architectural order is recognized by its voluted capital, featuring acanthus leaves and volutes similar to those of a ram's horn?
  [0] Corinthian order
  [1] Hellenistic order
  [2] Ionic order
  [3] Doric order
  --> Choice: "3" : "Doric order" (in 0.00s)
  ❌ WRONG | Earned: 0

=== Game over | Total earned: 0 ===
  [lo

## 5. LLM Strategy — Llama 3.2 (via Ollama)

Instead of guessing randomly, we use a local LLM running via Ollama. The model receives the question and four options, and must reply with only the correct letter (A, B, C or D). All inference runs locally — no external APIs are used.

In [67]:
# The model name, define we must.
MODEL_NAME = 'llama3.2'


def make_llm_strategy(prompt_fn):
    """Factory that wraps a prompt function into a strategy callable."""
    # A strategy factory, this is. Different prompts, same game loop.
    labels = ['A', 'B', 'C', 'D']
    def strategy(question):
        prompt = prompt_fn(question)
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = response['message']['content'].strip().upper()
        # The letter from the response, extract we must.
        chosen_label = next((c for c in raw if c in labels), None)
        if chosen_label is None:
            # No valid letter found — random fallback, use we will.
            print(f"  ⚠️ Unparseable response: '{raw[:60]}' — using random fallback.")
            return random.choice(question.options).id
        return question.options[labels.index(chosen_label)].id
    return strategy


print(f'LLM strategy factory ready. Model: {MODEL_NAME}')


LLM strategy factory ready. Model: llama3.2


## 6. Prompt Engineering Experiments

We compare two prompting strategies:

- **Zero-shot**: just the question and options, no examples
- **Chain-of-Thought (CoT)**: the model is asked to reason step by step before answering

In [68]:
# --- ZERO-SHOT ---

def prompt_zero_shot(question):
    # Simple and direct, the zero-shot prompt is.
    labels = ['A', 'B', 'C', 'D']
    options_text = '\n'.join(f'{labels[i]}) {opt.text}' for i, opt in enumerate(question.options))
    return (
        f'Answer the following multiple choice question. '
        f'Reply with ONLY the letter of the correct answer (A, B, C or D), nothing else.\n\n'
        f'Question: {question.text}\n\n'
        f'{options_text}\n\n'
        f'Answer:'
    )


# --- CHAIN-OF-THOUGHT ---

def prompt_cot(question):
    # Think step by step, the model must. Then answer, it will.
    labels = ['A', 'B', 'C', 'D']
    options_text = '\n'.join(f'{labels[i]}) {opt.text}' for i, opt in enumerate(question.options))
    return (
        f'Answer the following multiple choice question.\n'
        f"Think step by step, then write your final answer as 'Answer: X' where X is A, B, C or D.\n\n"
        f'Question: {question.text}\n\n'
        f'{options_text}\n'
    )


strategy_zero_shot = make_llm_strategy(prompt_zero_shot)
strategy_cot       = make_llm_strategy(prompt_cot)

print('Prompt strategies defined. Ready they are.')


Prompt strategies defined. Ready they are.


In [69]:
# Evaluated, both strategies will be.
N_GAMES_PROMPT = 5

df_zero = evaluate_strategy(client, COMPETITION_ID, strategy_zero_shot, 'zero-shot',        N_GAMES_PROMPT)
df_cot  = evaluate_strategy(client, COMPETITION_ID, strategy_cot,        'chain-of-thought', N_GAMES_PROMPT)



--- Game 1/5 ---

=== Strategy: zero-shot | Competition ID: 1 ===
Session ID: 87091

[Level 1] What is the fundamental principle of Classical Athens that made it a significant center of Western civilization?
  [0] Feudalism
  [1] Monarchy
  [2] Democracy
  [3] Slavery
  --> Choice: "2" : "Democracy" (in 1.99s)
  ✅ CORRECT | Earned: 100

[Level 2] In Homer's works, which term is one of the primary names used to refer to the Greeks as a whole, appearing 598 times in the Iliad?
  [0] Panhellenes
  [1] Danaans
  [2] Argives
  [3] Achaeans
  --> Choice: "0" : "Panhellenes" (in 0.19s)
  ❌ WRONG | Earned: 100

=== Game over | Total earned: 100 ===
  [log] game 1 saved → runs_log.json (6 records total)

--- Game 2/5 ---

=== Strategy: zero-shot | Competition ID: 1 ===
Session ID: 87093

[Level 1] Which class of inhabitants in Sparta was considered to be state-owned serfs and not full citizens?
  [0] Perioikoi
  [1] Helots
  [2] Trophimoioi
  [3] Spartiates
  --> Choice: "1" : "Helots" (in 0.1

## 7. Results Comparison

We compare all strategies across accuracy, average level reached, average earnings, and average response time.

In [70]:
# Compared, all strategies must be.
summary = pd.DataFrame([
    summarize(df_random),
    summarize(df_zero),
    summarize(df_cot),
])
print(summary.to_string(index=False))


        Strategy  Games  Total questions Accuracy Avg level Avg earned Avg time (s)
          random      5                5       0%       1.0          0        0.000
       zero-shot      5               26      28%       5.2       1020        0.258
chain-of-thought      5                5       0%       1.0          0        2.879


### Key findings

- **Zero-shot** clearly outperforms random, showing the LLM has strong general knowledge.
- **Chain-of-Thought** performs worse with a small model (3B parameters): reasoning step by step confuses rather than helps, and response time increases ~10×.
- Response times are well within the 30-second limit even for CoT.

#### Why Chain-of-Thought did not improve the results

Chain-of-Thought prompting is usually more effective with large language models, because they have enough capacity to follow a multi-step reasoning process. In our case, we used `llama3.2:3B`, which is a relatively small model. For this reason, asking the model to reason step by step often produced text that looked like reasoning, but was actually inconsistent or misleading.

Another issue is that the model can commit too early to a wrong interpretation. Once the first reasoning step is incorrect, the following steps tend to build on that mistake, making the final answer even more likely to be wrong. With zero-shot prompting, instead, the model directly predicts the answer without generating a long reasoning chain that may amplify errors.

Finally, Chain-of-Thought made the output format less predictable. In zero-shot prompting, the model usually returned a single option such as `A`, `B`, `C`, or `D`, which was easy to parse automatically. With Chain-of-Thought, the answer was often hidden inside a longer paragraph or sometimes not clearly stated at all. This caused parsing failures and activated the random fallback strategy, further reducing the reliability of the results.

## Setup API milionario


In [71]:
import numpy as np
from sentence_transformers import SentenceTransformer


EMBEDDER = SentenceTransformer('all-MiniLM-L6-v2')


class SemanticQuestionBank:
    """Stores question-answer pairs and retrieves the most similar ones via cosine similarity."""

    def __init__(self, embedder, min_examples=1, top_k=3):
        self.embedder    = embedder
        self.min_examples = min_examples
        self.top_k       = top_k
        self._pairs      = []          # list of (question_text, answer_text)
        self._embeddings = None        # (N, D) float32 array

    def add(self, question: str, answer: str):
        emb = self.embedder.encode([question], convert_to_numpy=True)
        self._pairs.append((question, answer))
        self._embeddings = emb if self._embeddings is None else np.vstack([self._embeddings, emb])

    def add_batch(self, pairs):
        if not pairs:
            return
        questions = [q for q, _ in pairs]
        embs = self.embedder.encode(questions, convert_to_numpy=True, show_progress_bar=True)
        self._pairs.extend(pairs)
        self._embeddings = embs if self._embeddings is None else np.vstack([self._embeddings, embs])

    def retrieve(self, query: str):
        if len(self._pairs) < self.min_examples:
            return []
        q_emb = self.embedder.encode([query], convert_to_numpy=True)   # (1, D)
        # Cosine similarity: dot product of normalised vectors.
        norms = np.linalg.norm(self._embeddings, axis=1, keepdims=True) + 1e-8
        q_norm = np.linalg.norm(q_emb) + 1e-8
        sims = (self._embeddings / norms) @ (q_emb / q_norm).T          # (N, 1)
        top_k = min(self.top_k, len(self._pairs))
        top_idx = np.argsort(sims[:, 0])[::-1][:top_k]
        return [self._pairs[i] for i in top_idx]

    def __len__(self):
        return len(self._pairs)


print(f'SemanticQuestionBank ready. Embedder: {EMBEDDER.get_sentence_embedding_dimension()}d embeddings.')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7503.49it/s]


SemanticQuestionBank ready. Embedder: 384d embeddings.


/var/folders/cr/4ss75rns1896p7fvnz3rgqnm0000gn/T/ipykernel_21853/2443036687.py:47: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'SemanticQuestionBank ready. Embedder: {EMBEDDER.get_sentence_embedding_dimension()}d embeddings.')


In [72]:
def preload_mmlu(bank, n_per_subject=25):
    """
    Pre-populate the bank with MMLU questions — no category labels used.
    Diverse topics loaded are, so any competition category covered will be.
    """
    subjects = [
        'high_school_physics', 'high_school_chemistry', 'high_school_biology',
        'high_school_mathematics', 'high_school_statistics', 'college_mathematics',
        'high_school_world_history', 'high_school_european_history', 'prehistory',
        'high_school_geography', 'sociology', 'high_school_macroeconomics',
        'astronomy', 'anatomy', 'nutrition', 'virology',
        'philosophy', 'logical_fallacies', 'high_school_computer_science',
        'high_school_psychology', 'global_facts', 'miscellaneous',
    ]
    labels = ['A', 'B', 'C', 'D']
    pairs  = []

    for subject in subjects:
        try:
            ds = load_dataset('cais/mmlu', subject, split='test')
            n  = min(n_per_subject, len(ds))
            for row in ds.select(range(n)):
                correct_idx = row['answer']
                answer_text = f"{labels[correct_idx]}) {row['choices'][correct_idx]}"
                pairs.append((row['question'], answer_text))
        except Exception as e:
            print(f'  ⚠️ Could not load {subject}: {e}')

    print(f'Loaded {len(pairs)} questions. Embedding them now...')
    bank.add_batch(pairs)
    print(f'Done. Bank size: {len(bank)}')


# Pre-populated with MMLU, the bank will be. No category labels, maximum generalisation.
semantic_bank = SemanticQuestionBank(EMBEDDER, min_examples=1, top_k=3)
preload_mmlu(semantic_bank, n_per_subject=25)


Loaded 550 questions. Embedding them now...


Batches: 100%|██████████| 18/18 [00:00<00:00, 44.84it/s]

Done. Bank size: 550


In [73]:
def make_semantic_few_shot_strategy(client, competition_id, bank):
    """
    Strategy using semantic retrieval for few-shot examples.
    Searches the whole bank regardless of category — maximum generalisation.
    Falls back to zero-shot when the bank has too few entries.
    """
    # The competition info, fetch once we must.
    comp   = get_competition_info(client, competition_id)
    labels = ['A', 'B', 'C', 'D']

    def strategy(question):
        options_text = '\n'.join(
            f'{labels[i]}) {opt.text}' for i, opt in enumerate(question.options)
        )

        
            # Similar examples retrieved from the whole bank, they will be.
        examples = bank.retrieve(question.text)
        examples_block = '\n'.join(
            f'Question: {q}\nAnswer: {a}\n' for q, a in examples
        )
        prompt = (
            f'Here are some similar questions with correct answers:\n\n'
            f'{examples_block}\n'
            f'Now answer this question. Reply with ONLY the letter (A, B, C or D).\n\n'
            f'Question: {question.text}\n'
            f'{options_text}\n'
            f'Answer:'
        )
        source = f'semantic-bank ({len(bank)} entries)'
    

        print(f'  [source: {source}]')

        response = ollama.chat(
            model=MODEL_NAME,
            messages=[{'role': 'user', 'content': prompt}]
        )
        raw = response['message']['content'].strip().upper()
        chosen_label = next((c for c in raw if c in labels), None)
        if chosen_label is None:
            print(f"  ⚠️ Unparseable: '{raw[:60]}' — random fallback.")
            return random.choice(question.options).id
        return question.options[labels.index(chosen_label)].id

    return strategy


print('Semantic few-shot strategy factory ready.')


Semantic few-shot strategy factory ready.


In [74]:
def evaluate_semantic_strategy(client, competition_id, bank, strategy_name, n_games=5):
    """Run multiple games, adding correct answers to the bank after each question."""
    comp     = get_competition_info(client, competition_id)
    all_logs = []

    for i in range(n_games):
        print(f'\n--- Game {i+1}/{n_games} | Bank size: {len(bank)} ---')
        strategy = make_semantic_few_shot_strategy(client, competition_id, bank)

        game = client.game.start(competition_id)
        print(f'Session ID: {game.session_id}')
        game_log = []

        while game.in_progress:
            q = game.current_question
            if not q:
                break

            print(f'\n[Level {game.current_level}] {q.text}')
            for opt in q.options:
                print(f'  [{opt.id}] {opt.text}')

            t_start    = time.time()
            chosen_id  = strategy(q)
            elapsed    = time.time() - t_start
            chosen_text = next(o.text for o in q.options if o.id == chosen_id)

            print(f'  --> Choice: "{chosen_text}" (in {elapsed:.2f}s)')
            result = game.answer(chosen_id)

            if result.correct:
                bank.add(q.text, chosen_text)

            row = {
                'game':            i + 1,
                'level':           game.current_level,
                'question':        q.text,
                'chosen':          chosen_text,
                'correct':         result.correct,
                'timed_out':       result.timed_out,
                'earned':          result.earned_amount,
                'strategy':        strategy_name,
                'response_time_s': round(elapsed, 3),
                'bank_size':       len(bank),
            }
            game_log.append(row)
            all_logs.append(row)

            status = '✅ CORRECT' if result.correct else ('⏰ TIMEOUT' if result.timed_out else '❌ WRONG')
            print(f'  {status} | Earned: {result.earned_amount}')

            if result.game_over:
                break

        append_to_log(game_log, competition_id, comp.name, i + 1, strategy_name)
        print(f'\n=== Game over | Earned: {game.earned_amount} | Bank: {len(bank)} ===')

    return pd.DataFrame(all_logs)


print('evaluate_semantic_strategy ready.')


evaluate_semantic_strategy ready.


In [75]:
# Run the semantic strategy on all 4 competitions.
# Shared bank used across all — cross-category generalisation, this enables.
N_GAMES = 5

df_sem_0 = evaluate_semantic_strategy(client, 0, semantic_bank, 'semantic-few-shot', N_GAMES)
df_sem_1 = evaluate_semantic_strategy(client, 1, semantic_bank, 'semantic-few-shot', N_GAMES)
df_sem_2 = evaluate_semantic_strategy(client, 2, semantic_bank, 'semantic-few-shot', N_GAMES)
df_sem_3 = evaluate_semantic_strategy(client, 3, semantic_bank, 'semantic-few-shot', N_GAMES)



--- Game 1/5 | Bank size: 550 ---
Session ID: 87103

[Level 1] What was the fundamental reason for U2's name change from 'Feedback' to 'The Hype' to 'U2'?
  [0] To reflect their evolving sound
  [1] To avoid confusion with another band
  [2] To make the name more catchy and marketable
  [3] To pay tribute to a fan's suggestion
  [source: semantic-bank (550 entries)]
  --> Choice: "To pay tribute to a fan's suggestion" (in 0.32s)
  ❌ WRONG | Earned: 0
  [log] game 1 saved → runs_log.json (16 records total)

=== Game over | Earned: 0 | Bank: 550 ===

--- Game 2/5 | Bank size: 550 ---
Session ID: 87104

[Level 1] Which actress is known for portraying the character Nazaré Tedesco in the Brazilian soap opera 'Senhora do Destino'?
  [0] Renata Sorrah
  [1] Susana Vieira
  [2] Adriana Esteves
  [3] Julia Roberts
  [source: semantic-bank (550 entries)]
  --> Choice: "Adriana Esteves" (in 0.29s)
  ❌ WRONG | Earned: 0
  [log] game 2 saved → runs_log.json (17 records total)

=== Game over | Earn

## 9. Results — All Competitions

We compare all strategies across all four competition categories. The random baseline and LLM strategies (zero-shot, CoT) were run on competition 0 (Entertainment). The semantic few-shot runs on all four competitions using the same shared bank.

In [76]:
# Per-competition summary for the semantic strategy, built we will.
def summarize_with_label(df, label):
    s = summarize(df)
    s['Competition'] = label
    return s

print('=== Baseline comparison (Competition 0 — Entertainment) ===')
baseline = pd.DataFrame([
    summarize(df_random),
    summarize(df_zero),
    summarize(df_cot),
    summarize(df_sem_0),
])
print(baseline.to_string(index=False))


=== Baseline comparison (Competition 0 — Entertainment) ===
         Strategy  Games  Total questions Accuracy Avg level Avg earned Avg time (s)
           random      5                5       0%       1.0          0        0.000
        zero-shot      5               26      28%       5.2       1020        0.258
 chain-of-thought      5                5       0%       1.0          0        2.879
semantic-few-shot      5               10       7%       2.0        100        0.307


In [77]:
# Semantic few-shot compared across all competitions.
print('\n=== Semantic Few-Shot — All Competitions ===')
cross = pd.DataFrame([
    summarize_with_label(df_sem_0, 'Entertainment'),
    summarize_with_label(df_sem_1, 'Ancient History & Politics'),
    summarize_with_label(df_sem_2, 'Science and Nature'),
    summarize_with_label(df_sem_3, 'Maths'),
])
# Competition column first, show we will.
cols = ['Competition'] + [c for c in cross.columns if c != 'Competition']
print(cross[cols].to_string(index=False))



=== Semantic Few-Shot — All Competitions ===
               Competition          Strategy  Games  Total questions Accuracy Avg level Avg earned Avg time (s)
             Entertainment semantic-few-shot      5               10       7%       2.0        100        0.307
Ancient History & Politics semantic-few-shot      5                8       4%       1.6         60        0.393
        Science and Nature semantic-few-shot      5               13      11%       2.6        180        0.345
                     Maths semantic-few-shot      5               10       7%       2.0        100        0.425


In [78]:
# Leggi e visualizza il log delle run salvate.
if LOG_FILE.exists():
    log_data = json.loads(LOG_FILE.read_text())
    df_log = pd.DataFrame(log_data)
    print(f'Log file: {LOG_FILE}  ({len(df_log)} records)\n')
    print(df_log.to_string(index=False))
else:
    print(f'Nessun log trovato in {LOG_FILE}. Esegui almeno una partita prima.')


Log file: /Users/aliceberta/Development/NLP/data/runs_log.json  (35 records)

          timestamp          strategy  competition_id             competition_name  game  correct  accuracy  earned  reached_level  response_time_avg
2026-05-11T17:38:09            random               1 Ancient History and Politics     1        0    0.0000     0.0              1              0.000
2026-05-11T17:38:09            random               1 Ancient History and Politics     2        0    0.0000     0.0              1              0.000
2026-05-11T17:38:09            random               1 Ancient History and Politics     3        0    0.0000     0.0              1              0.000
2026-05-11T17:38:10            random               1 Ancient History and Politics     4        0    0.0000     0.0              1              0.000
2026-05-11T17:38:10            random               1 Ancient History and Politics     5        0    0.0000     0.0              1              0.000
2026-05-11T17:38:12   